## 02.Row Filters and Column Masks — objective

- Compare raw fields with typed records and parsing failures.
- Explain how identity and governance affect visible data and permitted actions.

Historical results; no code executed.

# 02.Row Filters and Column Masks

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea09-data-security/02.Row%20Filters%20and%20Column%20Masks.json)

# Data-Level Security Using Row Filters and Column Masks

This notebook demonstrates how data-level security can be implemented directly on a table using row filters and column masks.

Unlike dynamic views, where logic is defined within a view, this approach applies security rules directly to the table. These rules are enforced automatically whenever the table is queried.

The implementation covers two key capabilities:

- Row-Level Security: Controls which rows are visible to a user  
- Column-Level Masking: Controls how sensitive values are displayed  

A simple sales dataset is used to illustrate how different users receive different results when querying the same table

> **Note:** Row filters and column masks are not supported on Dedicated (Single User) clusters as of now.
> Use a Shared cluster or a Serverless cluster to run this demo.

#### Step 1: Set up the schema for the table

Archived Databricks source — shown for reading, not Python execution.

```sql
USE CATALOG demo;
USE SCHEMA data_security;
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

#### Step 2: Create the table
A table is defined to represent sales data across multiple regions.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TABLE sales_secured (
  id INT,
  region STRING,
  email STRING,
  revenue INT
);

```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

#### Step 3: Insert Sample Data

Archived Databricks source — shown for reading, not Python execution.

```sql
INSERT INTO sales_secured VALUES
  (1, 'UK', 'john.doe@email.com', 10000),
  (2, 'US', 'mike.smith@email.com', 20000),
  (3, 'UK', 'sara.jones@email.com', 15000),
  (4, 'US', 'david.brown@email.com', 25000),
  (5, 'UK', 'emma.wilson@email.com', 18000);
```

In [ ]:
# Archived output only; no computation.

num_affected_rows,num_inserted_rows
5,5


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM sales_secured;
```

In [ ]:
# Archived output only; no computation.

id,region,email,revenue
1,UK,john.doe@email.com,10000
2,US,mike.smith@email.com,20000
3,UK,sara.jones@email.com,15000
4,US,david.brown@email.com,25000
5,UK,emma.wilson@email.com,18000


#### Step 4: Define Row Filter Function
The function evaluates the user’s group membership and returns a condition that determines whether a row should be included in the result.

For example:

- Users in the UK group are allowed to view UK records
- Users in the US group are allowed to view US records
- Admin users are allowed to view all records

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE FUNCTION fn_filter_region (region STRING)
RETURN 
  is_account_group_member('admin-sg')
  OR (is_account_group_member('uk-sg') AND region = 'UK')
  OR (is_account_group_member('us-sg') AND region = 'US');
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

#### Step 5: Apply Row Filter to Table
The row filter function is attached to the table. Once applied, the function is evaluated automatically at query time, and only the rows that satisfy the condition are returned.

Archived Databricks source — shown for reading, not Python execution.

```sql
ALTER TABLE sales_secured
SET ROW FILTER fn_filter_region ON (region);
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
DESC EXTENDED sales_secured;
```

In [ ]:
# Archived output only; no computation.

col_name,data_type,comment
id,int,null
region,string,null
email,string,null
revenue,int,null
,,
# Delta Statistics Columns,,
Column Names,"id, region, email, revenue",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM sales_secured;
```

In [ ]:
# Archived output only; no computation.

id,region,email,revenue
1,UK,john.doe@email.com,10000
3,UK,sara.jones@email.com,15000
5,UK,emma.wilson@email.com,18000


#### Step 6: Define Column Mask Function
The function evaluates the user’s group membership and determines whether to return the original value or a masked version.

For example:
- Admin users can view the full email address
- Other users see a masked version of the email

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE FUNCTION fn_mask_email (email STRING)
RETURN 
    CASE 
         WHEN is_account_group_member('admin-sg') THEN email
         ELSE concat(substr(email, 1, 1), '***@', split(email, '@')[1])
    END;
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

#### Step 7: Apply Column Mask
The masking function is attached to the email column. Once applied, the function is executed automatically whenever the column is queried.

Archived Databricks source — shown for reading, not Python execution.

```sql
ALTER TABLE sales_secured
ALTER COLUMN email SET MASK fn_mask_email;
```

In [ ]:
# Archived output only; no computation.

Archived rich output omitted. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
DESC EXTENDED sales_secured;
```

In [ ]:
# Archived output only; no computation.

col_name,data_type,comment
id,int,null
region,string,null
email,string,null
revenue,int,null
,,
# Delta Statistics Columns,,
Column Names,"id, region, email, revenue",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM sales_secured;
```

In [ ]:
# Archived output only; no computation.

id,region,email,revenue
1,UK,j***@email.com,10000
3,UK,s***@email.com,15000
5,UK,e***@email.com,18000
